Note: please open this file in Jupyter to get the same result as in the report.

# Question 2

In [1]:
# Group Formation Model

In [2]:
# import liabrary
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
import pandas as pd
from itertools import combinations
from pyomo.environ import value

## 2.1) Load data

In [3]:
file_name = 'AAMA_Group34_data.xlsx'

df = pd.read_excel(file_name, sheet_name='FIFA2026_qualified_nations',index_col=0)

nations = df.index.tolist()             # to get a list of 48 nations
points  = df['FIFA Points_24Feb26']     # FIFA points for nation i
conf    = df['Confederation']           # confederation for nation i
pot     = df['Pot_24Feb26']             # pot number for nation i
subset  = list(range(1,13))             # 12 subsets

# Pot and confederation groupings
pot_map  = {k: [i for i in nations if pot[i] == k] for k in [1, 2, 3, 4]}

confeds = sorted(conf.unique().tolist())
conf_map = {c: df.index[conf == c].tolist() for c in confeds}

## 2.2) Build Model - objective funtions / constraints

In [4]:
model = pyo.ConcreteModel()

model.x = pyo.Var(nations, subset, domain=pyo.Binary)
model.w = pyo.Var(domain=pyo.NonNegativeReals)

# Objective: to maximixe W (minimum total FIFA points for each subset)
model.obj = pyo.Objective(expr=model.w, sense=pyo.maximize)

# Fix each nation in Pot 1 to exactly one unique group.
pot1_nations = pot_map[1]
for i, nation_code in enumerate(pot1_nations):
    model.x[nation_code, subset[i]].fix(1)

# (1b): Minimum FIFA points
def rule_1b(model, g):
    return model.w <= sum(points[i] * model.x[i, g] for i in nations)
model.constr_1b = pyo.Constraint(subset, rule=rule_1b)

# (1c): 4 nations per subset
def rule_1c(model, g):
    return sum(model.x[i, g] for i in nations) == 4
model.constr_1c = pyo.Constraint(subset, rule=rule_1c)

# (1d): each nation in exactly one subset
def rule_1d(model, i):
    return sum(model.x[i, g] for g in subset) == 1
model.constr_1d = pyo.Constraint(nations, rule=rule_1d)

# (1e) A nation from any given pot per group
def rule_1e(model, g, pot_num):
    return sum(model.x[i, g] for i in pot_map[pot_num]) == 1
model.constr_1e = pyo.Constraint(subset, pot_map.keys(), rule=rule_1e)

# (1f) At most 1 nation from any given non-UEFA confederation per group
def rule_1f(model, g, cname):
    if cname == 'UEFA':
        return pyo.Constraint.Skip
    return sum(model.x[i, g] for i in conf_map[cname]) <= 1
model.constr_1f = pyo.Constraint(subset, conf_map.keys(), rule=rule_1f)

# (1g) At most 2 UEFA nations per subset
def rule_1g(model, g):
    return sum(model.x[i, g] for i in conf_map.get('UEFA', [])) <= 2
model.constr_1g = pyo.Constraint(subset, rule=rule_1g)

## 2.3) Solve by Solver

In [5]:
solver  = pyo.SolverFactory('glpk')
results = solver.solve(model, load_solutions=False, tee=False)

if (results.solver.status == SolverStatus.ok) and \
   (results.solver.termination_condition == TerminationCondition.optimal):
    model.solutions.load_from(results)
else:
    print("Solve failed.")
    exit(1)

## 2.4) Results

In [6]:
# 4. EXTRACT SOLUTION
assignment = {g: [i for i in nations if pyo.value(model.x[i, g]) > 0.5] for g in subset}

export = []
totals = []

print("=" * 65)
print(f" FIFA 2026 OPTIMAL GROUP FORMATION  |  w* = {pyo.value(model.w):,.2f} pts")
print("=" * 65)

for g in subset:
    members = sorted(assignment[g], key=lambda i: (pot[i], -points[i]))
    total   = sum(points[i] for i in members)
    totals.append(total)

    print(f"\n  Subset {g}")
    print(f"  {'Nation':<14}  {'Confederation':<14}  {'Pot':>3}  {'FIFA Points':>11}")
    print(f"  {'-'*47}")
    for i in members:
        print(f"  {i:<14}  {conf[i]:<14}  {pot[i]:>3}  {points[i]:>11,.2f}")
        export.append({"Subset": g, "FIFA Code": i, "Confederation": conf[i],
                       "Pot": pot[i], "FIFA Points": round(points[i], 2)})
    print(f"  {'─'*47}")
    print(f"  {'TOTAL':>36}  {total:>10,.2f}")

print("\n" + "=" * 65)
print(f"  Objective (w*) : {pyo.value(model.w):>10,.2f}")
print(f"  Min subset pts : {min(totals):>10,.2f}")
print(f"  Max subset pts : {max(totals):>10,.2f}")
print(f"  Range          : {max(totals)-min(totals):>10,.2f}")
print(f"  Average        : {sum(totals)/len(totals):>10,.2f}")
print("=" * 65)

 FIFA 2026 OPTIMAL GROUP FORMATION  |  w* = 6,337.03 pts

  Subset 1
  Nation          Confederation   Pot  FIFA Points
  -----------------------------------------------
  Spain           UEFA              1     1,877.18
  Ecuador         CONMEBOL          2     1,591.73
  Australia       AFC               3     1,574.01
  Curaçao         CONCACAF          4     1,302.70
  ───────────────────────────────────────────────
                                 TOTAL    6,345.62

  Subset 2
  Nation          Confederation   Pot  FIFA Points
  -----------------------------------------------
  Argentina       CONMEBOL          1     1,873.33
  South Korea     AFC               2     1,599.45
  Scotland        UEFA              3     1,506.77
  Cabo Verde      CAF               4     1,370.49
  ───────────────────────────────────────────────
                                 TOTAL    6,350.04

  Subset 3
  Nation          Confederation   Pot  FIFA Points
  ------------------------------------------

# Question 3

In [7]:
# MODEL 2 – GROUP-LETTER ASSIGNMENT MODEL

In [8]:
# 'groups' is built to automatically retrieve Model 1's solution
groups = assignment

## 3.1) Load data

In [9]:
df2 = pd.read_excel(file_name, sheet_name='Stadium Data', index_col=0)
df3 = pd.read_excel(file_name, sheet_name='Match_schedule', index_col=0)
spectator_index = df.loc[:,'Spectator Index']
row             = list(range(1,17))                                    # row-set index
n_matches       = df2.loc[:,'No. of matches assigned (Group Stage)']   # match number allocated to each stadium
letter          = list("ABCDEFGHIJKL")                                 # group letter index

fi = dict(zip(df.index, df['Spectator Index']))                        # read spectator index

In [10]:
# Rank nations within each subset
# Rank 1 = host nation (leads its subset by convention); others by descending FIFA points.
hosts = ["Mexico", "Canada", "United States"]
ranked_groups = {}
for g in subset:
    teams = groups[g]
    ranked = sorted(teams, key=lambda t: (t not in hosts, -points[t]))
    ranked_groups[g] = ranked

In [11]:
# define match set for each subset g
Mg = {} 
for g, team_list in ranked_groups.items():   
    Mg[g] = []
    n = len(team_list)
    for i in range(n):
        for j in range(i + 1, n):
            i1 = team_list[i]
            i2 = team_list[j]
            Mg[g].append((i1, i2))

# Match popularity π_{g,(i1,i2)} 
def match_popularity(fi1, fi2): 
    spectators_i1 = fi1
    spectators_i2 = fi2
    spectators_other = (fi1 + fi2) / 2.0
    officials = 1.0 if max(fi1, fi2) == 1.0 else (fi1 + fi2) / 2.0
    return spectators_i1 + spectators_i2 + spectators_other + officials

# define π_g(i1,i2)
pi = {}  
for g in Mg:
    for (i1, i2) in Mg[g]:
        fi1 = fi[i1]
        fi2 = fi[i2]
        pi[(g, i1, i2)] = match_popularity(fi1, fi2)

# Row-set structure J_r
# J[r] = list of (group_letter, rank_team1, rank_team2) for each match in row-set r
df3 = df3.reset_index()
J = {}
for stadium, group in df3.groupby('Stadium', sort=True):
    J[stadium] = list(zip(group['Group Letter'], group['team1'], group['team2']))

## 3.2) Build Model - objective funtions / constraints

In [12]:
model2 = pyo.ConcreteModel()

model2.z    = pyo.Var(subset, letter, domain=pyo.Binary)           # z_{g,l}
model2.y    = pyo.Var(row, domain=pyo.NonNegativeReals)            # y_r
model2.wmin = pyo.Var(domain=pyo.NonNegativeReals)                 # w^min
model2.wmax = pyo.Var(domain=pyo.NonNegativeReals)                 # w^max
model2.v    = pyo.Var(row, domain=pyo.Binary)                      # v_r

# Objective: maximise w^min + w^max (minimum row-set popularity + maximum row-set popularuty)
model2.obj = pyo.Objective(expr=model2.wmin + model2.wmax, sense=pyo.maximize)

# (2b): To set lowerbound for row-set popularity (w^min <= y_r   for all r)
def rule_2b(model2, r):
    return model2.wmin <= model2.y[r]
model2.constr_2b = pyo.Constraint(row, rule=rule_2b)

# (2c): To set Upperbound for row-set popularity (w^max >= y_r   for all r)
def rule_2c(model2, r):
    return model2.wmax >= model2.y[r]
model2.constr_2c = pyo.Constraint(row, rule=rule_2c)

# (2d): w^max <= y_r + M_r*(1 - v_r)   for all r
# M_r = 4 * n_matches[r]  (modified from paper; paper uses 16 for 4-match row-sets)
# the M_r is equivalent to a big-M which prevents wmax from being unbounded
def rule_2d(model2, r):
    return model2.wmax <= model2.y[r] + 4 * n_matches.iloc[r - 1] * (1 - model2.v[r])
model2.constr_2d = pyo.Constraint(row, rule=rule_2d)

# (2e): sum_{r} v_r = 1
# make sure that one row-set popularity exactly equals to the maximum row-set popularity(wmax)
model2.constr_2e = pyo.Constraint(expr=sum(model2.v[r] for r in row) == 1)

# (2f): row-set popularity computation (y_r = sum_{(l,q1,q2) in J_r} sum_{g} pop[g][(q1,q2)] * z[g,l]   for all r)
def rule_2f(model2, r):
    total_popularity = 0
    for (l, q1, q2) in J[r]:
        total_popularity += sum(pi[(g, ranked_groups[g][q1 - 1], ranked_groups[g][q2 - 1])] * model2.z[g,l] for g in subset)
    return model2.y[r] == total_popularity
model2.constr_2f = pyo.Constraint(row, rule=rule_2f)

# (2g): each letter assigned to exactly one subset (sum_{g} z[g,l] = 1   for all l) 
def rule_2g(model2, l):
    return sum(model2.z[g, l] for g in subset) == 1
model2.constr_2g = pyo.Constraint(letter, rule=rule_2g)

# (2h): each subset assigned to exactly one letter (sum_{l} z[g,l] = 1   for all g) 
def rule_2h(model2, g):
    return sum(model2.z[g, l] for l in letter) == 1
model2.constr_2h = pyo.Constraint(subset, rule=rule_2h)

# (2i): fix host-nation subsets to their designated letters
g_mex = None
g_can = None
g_us = None
for g, teams_in_g in groups.items():
    if 'Mexico' in teams_in_g:
        g_mex = int(g)
    if 'Canada' in teams_in_g:
        g_can = int(g)
    if 'United States' in teams_in_g:
        g_us = int(g)
model2.constr_2ia = pyo.Constraint(expr=model2.z[g_mex, 'A'] == 1)
model2.constr_2ib = pyo.Constraint(expr=model2.z[g_can, 'B'] == 1)
model2.constr_2ic = pyo.Constraint(expr=model2.z[g_us, 'D'] == 1)

## 3.3) Solve by Solver

In [13]:
solver2  = pyo.SolverFactory('glpk')
results2 = solver2.solve(model2, load_solutions=False, tee=False)

if (results2.solver.status == SolverStatus.ok) and \
   (results2.solver.termination_condition == TerminationCondition.optimal):
    model2.solutions.load_from(results2)
else:
    print("Solve failed.")
    exit(1)

## 3.4) Results

In [14]:
# Extract solution
zv    = pd.DataFrame(index=subset, columns=letter, dtype='Float64')
yv    = pd.Series(index=row, dtype='Float64')
wminv = pyo.value(model2.wmin)
wmaxv = pyo.value(model2.wmax)

for g in subset:
    for l in letter:
        zv.loc[g, l] = pyo.value(model2.z[g, l])
for r in row:
    yv[r] = pyo.value(model2.y[r])

oval2 = pyo.value(model2.obj)
print("Objective:", oval2)

subset_to_letter = {g: l for g in subset for l in letter if zv.loc[g, l] > 0.5}
letter_to_subset = {v: k for k, v in subset_to_letter.items()}

# RESULTS
# Result1: Group Letter Assignment
result_1 = pd.DataFrame([{
    'Group Letter': subset_to_letter[g],
    'Subset':       g,
    'Rank1':        ranked_groups[g][0],
    'Rank2':        ranked_groups[g][1],
    'Rank3':        ranked_groups[g][2],
    'Rank4':        ranked_groups[g][3]
} for g in subset]).sort_values('Group Letter').reset_index(drop=True)

print("\n" + "=" * 60)
print(f" GROUP-LETTER ASSIGNMENT  |  Objective = {oval2:.4f}")
print(f" w^min = {wminv:.4f}   |   w^max = {wmaxv:.4f}")
print("=" * 60)
print(f"\n  {'Group Letter':<14} {'Subset':<8} {'Rank1':<14} {'Rank2':<14} {'Rank3':<14} {'Rank4'}")
print(f"  {'─'*80}")
for _, rec in result_1.iterrows():
    print(f"  {rec['Group Letter']:<14} {rec['Subset']:<8} {rec['Rank1']:<14} "
          f"{rec['Rank2']:<14} {rec['Rank3']:<14} {rec['Rank4']}")

# Result2: Match Popularity
result_2_rows = []
for r in row:
    for (l, q1, q2) in J[r]:
        g = letter_to_subset[l]
        nation1 = ranked_groups[g][q1 - 1]
        nation2 = ranked_groups[g][q2 - 1]

        result_2_rows.append({
            'Row-set':            r,
            'Row-set Popularity': round(yv[r], 4),
            'Group Letter':       l,
            'Subset':             g,
            'Rank q1':            q1,
            'Nation 1':           nation1,
            'Rank q2':            q2,
            'Nation 2':           nation2,
            'Match Popularity':   round(pi[(g, nation1, nation2)], 4)
        })
result_2 = pd.DataFrame(result_2_rows)

print(f"\n  {'Row-set':>8} {'y_r':>10} {'Letter':>8} {'Rk1':>5} {'Nation1':<14} "
      f"{'Rk2':>5} {'Nation2':<14} {'Popularity':>12}")
print(f"  {'─'*85}")
for _, rec in result_2.iterrows():
    print(f"  {rec['Row-set']:>8} {rec['Row-set Popularity']:>10.4f} "
          f"{rec['Group Letter']:>8} {rec['Rank q1']:>5} {rec['Nation 1']:<14} "
          f"{rec['Rank q2']:>5} {rec['Nation 2']:<14} {rec['Match Popularity']:>12.4f}")

# Result3: Row-set Popularity Summary
result_3 = pd.DataFrame([{
    'Row-set':            r,
    'Number of Matches':  n_matches[r-1],
    'Row-set Popularity': round(yv[r], 4)
} for r in row])

print(f"\n  {'Row-set':>8} {'# Matches':>12} {'Row-set Popularity':>20}")
print(f"  {'─'*44}")
for _, rec in result_3.iterrows():
    print(f"  {rec['Row-set']:>8} {rec['Number of Matches']:>12} "
          f"{rec['Row-set Popularity']:>20.4f}")
print(f"  {'─'*44}")
print(f"  {'w^min':>32}  {wminv:.4f}")
print(f"  {'w^max':>32}  {wmaxv:.4f}")
print(f"  {'Objective (w^min+w^max)':>32}  {oval2:.4f}")

Objective: 30.072243836434602

 GROUP-LETTER ASSIGNMENT  |  Objective = 30.0722
 w^min = 10.0722   |   w^max = 20.0000

  Group Letter   Subset   Rank1          Rank2          Rank3          Rank4
  ────────────────────────────────────────────────────────────────────────────────
  A              11       Mexico         Switzerland    Algeria        Uzbekistan
  B              12       Canada         Germany        Austria        DR Congo
  C              4        England        Colombia       Egypt          New Zealand
  D              10       United States  Uruguay        Ukraine        Iraq
  E              8        Morocco        Denmark        Norway         Qatar
  F              7        Netherlands    IR Iran        Panama         South Africa
  G              5        Brazil         Senegal        Turkey         Haiti
  H              6        Portugal       Italy          Tunisia        Saudi Arabia
  I              3        France         Japan          Paraguay       Ghana


C:\Users\pun\AppData\Local\Temp\ipykernel_38144\2898708071.py:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Number of Matches':  n_matches[r-1],


# Question 4

In [15]:
# MODEL 3 – STADIUM ASSIGNMENT MODEL

## 4.1) Load data

In [16]:
# Load stadium data (tab: 'Stadium Data')
df_stad3     = pd.read_excel(file_name, sheet_name='Stadium Data',
                              usecols=['Stadium', 'Capacity', 'Country',
                                       'No. of matches assigned (Group Stage)'])
stadiums     = df_stad3['Stadium'].tolist()
capacity     = df_stad3.set_index('Stadium')['Capacity'].to_dict()                               # κ_s: stadium capacity
country      = df_stad3.set_index('Stadium')['Country'].to_dict()                                # country of stadium s
stad_matches = df_stad3.set_index('Stadium')['No. of matches assigned (Group Stage)'].to_dict()  # number of matches assigned to each stadium

In [17]:
# Identify row-sets with host-nation matches and their required country
# To identify which row-sets contain 3 host nations : MEX (Group A, Rank 1) -> Mexico; CAN (Group B, Rank 1) -> Canada; USA (Group D, Rank 1)
host_requirements = {'A': 'Mexico', 'B': 'Canada', 'D': 'United States'}

rowset_required_country = {}
for r in J:
    for grp_letter, hostcountry in host_requirements.items():
        if any(l == grp_letter and (q1 == 1 or q2 == 1) for (l, q1, q2) in J[r]):
            rowset_required_country[r] = hostcountry
            break

## 4.2) Build Model - objective funtions / constraints

In [18]:
model3 = pyo.ConcreteModel()

model3.u = pyo.Var(row, stadiums, domain=pyo.Binary)   # u[r,s] = 1 iff row-set r -> stadium s

# Objective: maximise sum_{r,s} κ_s * ȳ_r * u[r,s]
model3.obj = pyo.Objective(expr=sum(capacity[s] * yv[r] * model3.u[r, s] for r in row for s in stadiums), sense=pyo.maximize)

# (3a): each row-set assigned to exactly one stadium
def rule_3a(model3, r):
    return sum(model3.u[r, s] for s in stadiums) == 1
model3.constr_3a = pyo.Constraint(row, rule=rule_3a)

# (3b): each stadium assigned to exactly one row-set
def rule_3b(model3, s):
    return sum(model3.u[r, s] for r in row) == 1
model3.constr_3b = pyo.Constraint(stadiums, rule=rule_3b)

# (3c): host-nation country constraints
# For every restricted row-set r, fix u[r,s] = 0 for stadiums outside the required country.
for r, req_country in rowset_required_country.items():
    for s in stadiums:
        if country[s] != req_country:
            model3.u[r, s].fix(0)

# (3d): match-count parity
# Row-set r may only be assigned to stadium s if both have the same number of matches.
# Fix u[r,s] = 0 for all (r,s) pairs where n_matches[r] != stad_matches[s].
for r in row:
    for s in stadiums:
        if n_matches[r-1] != stad_matches[s]:
            model3.u[r, s].fix(0)

C:\Users\pun\AppData\Local\Temp\ipykernel_38144\1859117619.py:30: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if n_matches[r-1] != stad_matches[s]:


## 4.3) Solve by Solver

In [19]:
solver3  = pyo.SolverFactory('glpk')
results3 = solver3.solve(model3, load_solutions=False, tee=False)

if (results3.solver.status == SolverStatus.ok) and \
   (results3.solver.termination_condition == TerminationCondition.optimal):
    model3.solutions.load_from(results3)
else:
    print("Solve failed.")
    exit(1)

## 4.4) Results

In [20]:
# Extract solution
oval3 = pyo.value(model3.obj)
print("Objective:", oval3)

rowset_to_stadium = {r: s for r in row for s in stadiums
                     if pyo.value(model3.u[r, s]) > 0.5}

print("\n" + "=" * 70)
print(f" STADIUM ASSIGNMENT  |  Objective = {oval3:,.2f}")
print("=" * 70)
print(f"\n  {'Row-set':>8}  {'Stadium':<35}  {'Country':<15}  {'Capacity':>10}  {'Popularity':>12}")
print(f"  {'─'*90}")
for r in row:
    s = rowset_to_stadium[r]
    print(f"  {r:>8}  {s:<35}  {country[s]:<15}  {capacity[s]:>10,}  {yv[r]:>12.4f}")

Objective: 14661342.423301434

 STADIUM ASSIGNMENT  |  Objective = 14,661,342.42

   Row-set  Stadium                              Country            Capacity    Popularity
  ──────────────────────────────────────────────────────────────────────────────────────────
         1  Los Angeles Stadium                  United States        69,650       17.5774
         2  Atlanta Stadium                      United States        67,382       15.0937
         3  Seattle Stadium                      United States        65,123       13.1339
         4  Dallas Stadium                       United States        70,122       17.6029
         5  San Francisco Bay Area Stadium       United States        69,391       17.0727
         6  Philadelphia Stadium                 United States        65,827       14.9148
         7  Kansas City Stadium                  United States        67,513       14.4678
         8  Boston Stadium                       United States        63,815       11.5037
      